# Environment Setup

In [62]:
!pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [63]:
# Cell 1: Environment Setup
# %matplotlib inline # Uncomment if running in a different environment

!pip -q install einops timm torchmetrics  # already in most fastMRI envs

import os, math, time, csv, random, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, RandomSampler
from einops import rearrange
from skimage.metrics import structural_similarity as ssim # Renamed to avoid conflict
from torch.amp import GradScaler # For the device-agnostic GradScaler
from torch.amp import autocast   # For the device-agnostic autocast

# Set random seeds for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Using device: cuda


# Dataset Class Definition

In [64]:
# Cell 2: Dataset Class Definition
class ProcessedFastMRIDataset(Dataset):
    """Dataset for loading preprocessed FastMRI data or creating from raw files"""
    
    def __init__(self, data_dir=None, file_list=None, mode='train', mask_func=None, use_processed=True):
        self.mode = mode
        self.use_processed = use_processed
        
        if use_processed:
            self.data_dir = os.path.join(data_dir, mode)
            try:
                all_files_in_dir = os.listdir(self.data_dir)
            except FileNotFoundError:
                raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

            all_pt_files = sorted([os.path.join(self.data_dir, f) for f in all_files_in_dir if f.endswith('.pt')])

            self.batch_files = []
            for f_path_str in all_pt_files:
                if "metadata" not in Path(f_path_str).name:
                    self.batch_files.append(f_path_str)
                # else:
                #     print(f"Filtering out metadata file: {f_path_str}") # Can be verbose
            
            if not self.batch_files:
                raise FileNotFoundError(f"No valid .pt files (after filtering 'metadata' files) found in {self.data_dir}")
                
            self.examples = []
            # print(f"Found {len(self.batch_files)} .pt files to process in {self.data_dir} after filtering.")
            
            for i, batch_file_path in enumerate(self.batch_files):
                # print(f"Processing file: {batch_file_path}") # Can be verbose
                try:
                    # Just to get the number of samples, peek at one key or load metadata if available
                    # For simplicity, we load the 'inputs' key length, assuming it's representative
                    # This part can be slow if files are large. Consider storing metadata separately.
                    batch_peek = torch.load(batch_file_path, map_location='cpu') 

                    if not isinstance(batch_peek, dict):
                        print(f"  Warning: Skipped {batch_file_path}. Loaded object is not a dictionary (type: {type(batch_peek)}).")
                        continue
                    if 'inputs' not in batch_peek:
                        print(f"  Warning: Skipped {batch_file_path}. Missing 'inputs' key. Keys present: {list(batch_peek.keys())}.")
                        continue
                    
                    num_samples = len(batch_peek['inputs'])
                    self.examples.extend([(i, j) for j in range(num_samples)])
                    del batch_peek # Free memory

                except Exception as e:
                    print(f"  Error peaking into file {batch_file_path}: {e}. Skipping.")
            
            if not self.examples:
                raise ValueError(f"No valid examples could be loaded from {self.data_dir}. Check warnings above.")
            
            print(f"Successfully indexed a total of {len(self.examples)} examples from {len(self.batch_files)} .pt files in {self.data_dir}.")

        else:
            # This part is for use_processed=False, ensure it's complete from your original code
            # For this example, focusing on use_processed=True
            raise NotImplementedError("Raw file processing not fully set up in this tuning script example.")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        if self.use_processed:
            batch_idx, sample_idx = self.examples[idx]
            batch_data = torch.load(self.batch_files[batch_idx], map_location='cpu')
            inputs = batch_data['inputs'][sample_idx]
            targets = batch_data['targets'][sample_idx]
            del batch_data # Free memory
        else:
            raise NotImplementedError("Raw file processing not fully set up in this tuning script example.")
        
        return inputs, targets

# Loss Functions and Metrics

In [65]:
# Cell 3: Loss Functions and Metrics
class SSIMLoss(nn.Module):
    """SSIM loss module for MRI reconstruction"""
    def __init__(self, win_size=7, k1=0.01, k2=0.03):
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.register_buffer('w', torch.ones(1, 1, win_size, win_size) / win_size**2)
        self.cov_norm = win_size**2 / (win_size**2 - 1)
    
    def forward(self, x, y): # Expects x, y to be 4D tensors [B, C, H, W]
        data_range = 1.0  # Images are normalized to [0,1]
        C1 = (self.k1 * data_range)**2
        C2 = (self.k2 * data_range)**2
        
        # Add channel dim if not present (e.g., if input is [B, H, W])
        if x.ndim == 3: x = x.unsqueeze(1)
        if y.ndim == 3: y = y.unsqueeze(1)

        ux = F.conv2d(x, self.w)
        uy = F.conv2d(y, self.w)
        
        uxx = F.conv2d(x * x, self.w)
        uyy = F.conv2d(y * y, self.w)
        uxy = F.conv2d(x * y, self.w)
        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)
        
        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2
        
        D = (A1 * A2) / (B1 * B2)
        return 1 - D.mean()

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.84):
        super().__init__()
        self.alpha = alpha
        self.l1_loss = nn.L1Loss()
        self.ssim_loss = SSIMLoss()
        
    def forward(self, pred, target):
        l1 = self.l1_loss(pred, target)
        ssim = self.ssim_loss(pred, target)
        return self.alpha * l1 + (1 - self.alpha) * ssim

def calculate_psnr_torch(img1, img2): # Renamed to avoid conflict if calculate_psnr means numpy
    """Calculate PSNR between two PyTorch tensors"""
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0: return float('inf') # Perfect match
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

def calculate_ssim_numpy(img1_np, img2_np): # Explicitly for numpy arrays
    """Calculate SSIM between two numpy arrays"""
    return ssim(img1_np, img2_np, data_range=img1_np.max())

# Common Model Blocks

In [66]:
# %% [code]  Basic conv helpers used by every architecture
class DoubleConv(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU()
        )
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__(); self.pool=nn.AvgPool2d(2); self.conv=DoubleConv(in_ch,out_ch)
    def forward(self,x): return self.conv(self.pool(x))
import torch.nn.functional as F

class Up(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__(); self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        self.conv=DoubleConv(in_ch,out_ch)
    # --- Up.forward (final, robust) ------------------------------------------
    def forward(self, x, skip):
        x = self.up(x)
    
        # 1) spatial size guard (already added earlier)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                              align_corners=False)
    
        # 2) channel-count guard – crop the *larger* tensor
        c = min(x.shape[1], skip.shape[1])     # desired channels after crop
        if x.shape[1] != c:
            x = x[:, :c, ...]                  # trim x if it is larger
        if skip.shape[1] != c:
            skip = skip[:, :c, ...]            # trim skip if it is larger
    
        return self.conv(torch.cat([x, skip], dim=1))
    



# Transformer Primitives

In [67]:
# %% [code]  Transformer helpers
class CustomMLP(nn.Sequential):
    def __init__(self,dim,mlp_dim=None,p=0.):
        mlp_dim = mlp_dim or dim*4
        super().__init__(nn.Linear(dim,mlp_dim),nn.GELU(),nn.Dropout(p),
                         nn.Linear(mlp_dim,dim),nn.Dropout(p))
# In Cell 6
class TransformerBlock(nn.Module):
    def __init__(self,dim,heads=8,p=0.):
        super().__init__()
        self.n1=nn.LayerNorm(dim); self.attn=nn.MultiheadAttention(dim,heads,dropout=p,batch_first=True)
        # Changed MLP to CustomMLP here
        self.n2=nn.LayerNorm(dim); self.mlp=CustomMLP(dim,p=p)
    def forward(self,x):
        x=x+self.attn(self.n1(x),self.n1(x),self.n1(x))[0]
        return x+self.mlp(self.n2(x))


# Swin-UNet

In [68]:
# Cell 5: Swin-UNet Architecture
from timm.models.swin_transformer import WindowAttention, Mlp as TimmMLP # Use a distinct name

def window_partition(x, window_size: int):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size: int, H: int, W: int):
    num_windows_h = H // window_size
    num_windows_w = W // window_size
    B = windows.shape[0] // (num_windows_h * num_windows_w)
    x = windows.view(B, num_windows_h, num_windows_w, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x
    
class SwinBlock(nn.Module):
    def __init__(self, dim, ws=8, heads=4, drop_path_rate=0.): # Added drop_path_rate
        super().__init__()
        self.dim = dim
        self.window_size = ws
        self.heads = heads

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim,
            num_heads=heads,
            window_size=(ws, ws),
            qkv_bias=True
        )
        # DropPath can be added here if desired: self.drop_path1 = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = TimmMLP(in_features=dim, hidden_features=int(dim * 4), act_layer=nn.GELU, drop=0.) # TimmMLP uses 'drop' for dropout
        # DropPath can be added here: self.drop_path2 = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()


    def forward(self, x): # Expected input x: (B, C, H, W)
        B, C, H, W = x.shape
        if C != self.dim:
            raise ValueError(f"Input channel dimension {C} does not match block dimension {self.dim}")

        x_spatial = rearrange(x, 'b c h w -> b h w c')
        shortcut = x_spatial 

        x_normed = self.norm1(x_spatial)

        H_pad, W_pad = H, W
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x_normed = F.pad(x_normed, (0, 0, pad_l, pad_r, pad_t, pad_b)) 
            H_pad += pad_b
            W_pad += pad_r
        
        x_windows = window_partition(x_normed, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # W-MSA
        # For SW-MSA, cyclic shift and attention mask would be needed.
        attn_windows = self.attn(x_windows, mask=None)
        
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        merged_windows = window_reverse(attn_windows, self.window_size, H_pad, W_pad)

        if pad_r > 0 or pad_b > 0:
            merged_windows = merged_windows[:, :H, :W, :].contiguous()
        
        # x_spatial = shortcut + self.drop_path1(merged_windows) # If using DropPath
        x_spatial = shortcut + merged_windows
        
        x_ffn_input = self.norm2(x_spatial)
        x_ffn = self.mlp(x_ffn_input)
        
        # x_out_spatial = x_spatial + self.drop_path2(x_ffn) # If using DropPath
        x_out_spatial = x_spatial + x_ffn

        x_out = rearrange(x_out_spatial, 'b h w c -> b c h w')
        
        return x_out

class SwinUNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, base=32, ws=8, heads_per_stage=(4,4,4,4)): # Added heads_per_stage
        super().__init__()
        chs=[base, base*2, base*4, base*8] # e.g., 32, 64, 128, 256 if base=32
        self.inc=nn.Conv2d(in_chans,chs[0],3,padding=1) # Initial convolution
        
        # Encoder stages
        self.s1=self._stage(chs[0],chs[0],ws, heads=heads_per_stage[0])
        self.d1=nn.Conv2d(chs[0],chs[1],kernel_size=2,stride=2); self.s2=self._stage(chs[1],chs[1],ws, heads=heads_per_stage[1])
        self.d2=nn.Conv2d(chs[1],chs[2],kernel_size=2,stride=2); self.s3=self._stage(chs[2],chs[2],ws, heads=heads_per_stage[2])
        self.d3=nn.Conv2d(chs[2],chs[3],kernel_size=2,stride=2); self.s4=self._stage(chs[3],chs[3],ws, heads=heads_per_stage[3]) # Bottleneck stage
        
        # Decoder stages
        self.u3=nn.ConvTranspose2d(chs[3],chs[2],kernel_size=2,stride=2); self.su3=self._stage(chs[2]*2,chs[2],ws, heads=heads_per_stage[2]) # input is cat(u3_out, s3_skip)
        self.u2=nn.ConvTranspose2d(chs[2],chs[1],kernel_size=2,stride=2); self.su2=self._stage(chs[1]*2,chs[1],ws, heads=heads_per_stage[1])
        self.u1=nn.ConvTranspose2d(chs[1],chs[0],kernel_size=2,stride=2); self.su1=self._stage(chs[0]*2,chs[0],ws, heads=heads_per_stage[0])
        
        self.out=nn.Conv2d(chs[0],out_chans,kernel_size=1) # Output convolution

    def _stage(self,in_ch,out_ch,ws,heads): # Added heads parameter
        # A stage typically consists of a CNN block followed by a SwinTransformer block.
        # Here, it's a Conv -> Norm -> GELU -> SwinBlock sequence.
        return nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size=3,padding=1,bias=False),
            nn.InstanceNorm2d(out_ch,affine=True),
            nn.GELU(),
            SwinBlock(dim=out_ch, ws=ws, heads=heads) # Pass heads to SwinBlock
        )

    def forward(self,x):
        x0 = self.inc(x) # Initial conv
        s1 = self.s1(x0) # Stage 1 + skip
        
        s2_in = self.d1(s1) # Downsample
        s2 = self.s2(s2_in) # Stage 2 + skip
        
        s3_in = self.d2(s2) # Downsample
        s3 = self.s3(s3_in) # Stage 3 + skip
        
        b_in = self.d3(s3)  # Downsample to bottleneck
        b = self.s4(b_in)   # Bottleneck stage
        
        u3_up = self.u3(b)  # Upsample
        u3 = self.su3(torch.cat([u3_up, s3], dim=1)) # Concatenate skip and process
        
        u2_up = self.u2(u3) # Upsample
        u2 = self.su2(torch.cat([u2_up, s2], dim=1)) # Concatenate skip and process
        
        u1_up = self.u1(u2) # Upsample
        u1 = self.su1(torch.cat([u1_up, s1], dim=1)) # Concatenate skip and process
        
        return self.out(u1)

# Generic Trainer

In [69]:
# %% [code]  train_and_eval one model
def run_model(run_name, model_fn, train_ds, val_ds,
              batch_size=16, epochs=50, lr=1e-4, frac=0.25,
              out_root="./runs", device='cuda'):
    out_dir = Path(out_root)/run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    train_loader = make_loader(train_ds,batch_size,frac,shuffle=True)
    val_loader   = make_loader(val_ds,  batch_size,1.0, shuffle=False)

    model = model_fn().to(device)
    criterion = CombinedLoss(alpha=0.84).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,'min',0.5,5)

    best = float('inf')
    log = []

    for ep in range(1,epochs+1):
        print(f"\n{run_name} | epoch {ep}/{epochs}")
        tr_loss = train_epoch(model,train_loader,optimizer,criterion,device)
        vl_loss, vl_psnr, vl_ssim = validate(model,val_loader,criterion,device)
        scheduler.step(vl_loss)
        log.append([ep,tr_loss,vl_loss,vl_psnr,vl_ssim])

        # save best
        if vl_loss < best:
            best = vl_loss
            torch.save({'epoch':ep,'state':model.state_dict()}, out_dir/'best.pth')

        # light checkpoint every 10
        if ep%10==0:
            torch.save({'epoch':ep,'state':model.state_dict()}, out_dir/f'ckpt_{ep}.pth')

    # save log
    with open(out_dir/'history.csv','w',newline='') as f:
        w=csv.writer(f); w.writerow(['epoch','train','val','psnr','ssim']); w.writerows(log)
    return out_dir


# Visualization Functions

In [70]:
# Cell 6: Training, Validation, and Visualization Functions

def train_epoch(model, dataloader, optimizer, criterion, device, scaler, accumulation_steps=1):
    model.train()
    running_loss = 0.0
    
    with tqdm(dataloader, desc="Training", leave=False) as pbar:
        for i, (inputs, targets) in enumerate(pbar):
            if i % accumulation_steps == 0: 
                optimizer.zero_grad()

            inputs = inputs.to(device)
            targets = targets.to(device)

            if len(inputs.shape) == 3: inputs = inputs.unsqueeze(1)
            if len(targets.shape) == 3: targets = targets.unsqueeze(1)

            with autocast(device_type=device.type, enabled=scaler.is_enabled()): 
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                if accumulation_steps > 1:
                    loss = loss / accumulation_steps 

            scaler.scale(loss).backward() 

            if (i + 1) % accumulation_steps == 0: 
                scaler.step(optimizer)
                scaler.update()

            current_loss = loss.item() * (accumulation_steps if accumulation_steps > 1 and (i + 1) % accumulation_steps == 0 else 1)
            running_loss += current_loss
            pbar.set_postfix({'loss': current_loss})
            
    # Handle any remaining steps if dataloader size is not a multiple of accumulation_steps
    if len(dataloader) % accumulation_steps != 0:
        scaler.step(optimizer) 
        scaler.update()
        # optimizer.zero_grad() # Will be zeroed at start of next epoch

    return running_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_psnr = 0.0
    all_ssims = [] # Collect all SSIMs for averaging
    
    with torch.no_grad():
        with tqdm(dataloader, desc="Validation", leave=False) as pbar:
            for inputs, targets in pbar:
                inputs = inputs.to(device)
                targets = targets.to(device)
                
                if len(inputs.shape) == 3: inputs = inputs.unsqueeze(1)
                if len(targets.shape) == 3: targets = targets.unsqueeze(1)
                
                # No autocast needed for validation if not training
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                psnr = calculate_psnr_torch(outputs, targets) # Use torch version for consistency on device
                
                running_loss += loss.item()
                running_psnr += psnr.item()
                
                # Calculate SSIM on CPU for each item in batch
                for k in range(outputs.size(0)):
                    output_np = outputs[k, 0].cpu().numpy()
                    target_np = targets[k, 0].cpu().numpy()
                    all_ssims.append(calculate_ssim_numpy(output_np, target_np))
                    
                pbar.set_postfix({'val_loss': loss.item(), 'psnr': psnr.item()})
                
    avg_ssim = np.mean(all_ssims) if all_ssims else 0
    return running_loss / len(dataloader), running_psnr / len(dataloader), avg_ssim


def visualize_results(model, dataloader, device, epoch, output_dir_str):
    output_dir = Path(output_dir_str)
    output_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    
    inputs, targets = next(iter(dataloader))
    inputs = inputs.to(device)
    targets = targets.to(device)
    
    with torch.no_grad():
        outputs = model(inputs.unsqueeze(1) if len(inputs.shape) == 3 else inputs)
    
    fig, axes = plt.subplots(min(4, inputs.size(0)), 3, figsize=(15, 5 * min(4, inputs.size(0))))
    if min(4, inputs.size(0)) == 1: axes = np.array([axes]) # Ensure axes is 2D for single image
    fig.suptitle(f"SwinUNet Reconstruction - Epoch {epoch}", fontsize=16)
    
    for i in range(min(4, inputs.size(0))): 
        input_img = inputs[i].cpu().squeeze().numpy()
        output_img = outputs[i, 0].cpu().squeeze().numpy()
        target_img = targets[i].cpu().squeeze().numpy()
            
        axes[i, 0].imshow(input_img, cmap='gray')
        axes[i, 0].set_title(f"Input (Undersampled)")
        axes[i, 0].axis('off')
            
        axes[i, 1].imshow(output_img, cmap='gray')
        axes[i, 1].set_title(f"Prediction")
        axes[i, 1].axis('off')
            
        axes[i, 2].imshow(target_img, cmap='gray')
        axes[i, 2].set_title(f"Ground Truth")
        axes[i, 2].axis('off')
            
        psnr_val = calculate_psnr_torch(
            torch.from_numpy(output_img).unsqueeze(0).unsqueeze(0), 
            torch.from_numpy(target_img).unsqueeze(0).unsqueeze(0)
        ).item()
        ssim_val = calculate_ssim_numpy(output_img, target_img)
            
        axes[i, 1].text(
            5, 15, 
            f'PSNR: {psnr_val:.2f}\nSSIM: {ssim_val:.4f}',
            color='white', fontsize=10, 
            bbox=dict(facecolor='black', alpha=0.5)
        )
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(output_dir / f"epoch_{epoch}.png")
    plt.close(fig)

# run_model definition will be in the next cell (main script)

In [71]:
from pathlib import Path # Ensure Path is imported in this cell or globally

# ... (assuming _orig_init is correctly defined from the non-patched ProcessedFastMRIDataset.__init__)

def _patched_init(self, *args, **kwargs):
    _orig_init(self, *args, **kwargs)

    good_examples = []
    if hasattr(self, 'batch_files') and self.batch_files: # Check if batch_files exists and is not empty
        for idx, slice_idx in self.examples:
            # Ensure idx is a valid index for self.batch_files
            if 0 <= idx < len(self.batch_files):
                batch_file_string_path = self.batch_files[idx]
                # Convert the string path to a Path object
                path_obj = Path(batch_file_string_path)
                if "metadata" not in path_obj.name:  # Now path_obj.name is correct
                    good_examples.append((idx, slice_idx))
            else:
                print(f"Warning in patch: Invalid index {idx} for batch_files of length {len(self.batch_files)}")
        self.examples = good_examples
    elif not hasattr(self, 'batch_files') or not self.batch_files:
        # This case might occur if _orig_init failed to populate self.batch_files
        # or if use_processed was False (though the error trace suggests use_processed=True)
        print("Warning in patch: self.batch_files not populated or empty, skipping example filtering.")
        # self.examples would remain as whatever _orig_init set it to, or cause error if not set.

# ProcessedFastMRIDataset.__init__ = _patched_init # This line applies the patch
# print("Patched ProcessedFastMRIDataset to ignore files that contain 'metadata'")


# Main Training Script

In [72]:
# Cell 7: Main Hyperparameter Tuning Script for Swin-UNet
import pandas as pd # For results display

def run_tuning_experiment(run_name, model_fn, train_ds, val_ds,
                          batch_size=16, epochs=50, lr=1e-4, frac=1.0,
                          viz_every=10, out_root_str="./runs_swin_tuning", 
                          device_str='cuda', accumulation_steps=1):

    out_dir = Path(out_root_str) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    current_device = torch.device(device_str)

    def make_loader_local(ds, shuffle, fraction_local=1.0): # Renamed to avoid conflict
        if fraction_local < 1.0 and len(ds) > 0 : # Added len(ds) > 0 check
            n = int(len(ds) * fraction_local)
            if n == 0 and len(ds) > 0: n = 1 # Ensure at least one sample if dataset is not empty
            idx = random.sample(range(len(ds)), n) if len(ds) > n else list(range(len(ds)))
            ds_subset = Subset(ds, idx)
        else:
            ds_subset = ds
        
        if len(ds_subset) == 0:
            print(f"Warning: DataLoader for {'train' if shuffle else 'val'} created with 0 samples.")
            # Return an empty loader or handle as an error, depending on desired behavior
            return DataLoader(ds_subset, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


        return DataLoader(ds_subset, batch_size=batch_size,
                          shuffle=shuffle, num_workers=2, pin_memory=True, drop_last=(True if shuffle and len(ds_subset) > batch_size else False) )


    train_loader = make_loader_local(train_ds, True,  frac)
    val_loader   = make_loader_local(val_ds,   False, 1.0) # Validate on full validation set

    model     = model_fn().to(current_device)
    criterion = CombinedLoss(alpha=0.84).to(current_device) # As used in pilot
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4) # AdamW from pilot
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)
    scaler = GradScaler(device=current_device.type, enabled=current_device.type=='cuda')


    best_val_loss = float('inf')
    history = [] 

    print(f"\n▶ Starting Run: {run_name} – Training on {len(train_loader.dataset)} slices ({frac*100:.0f}% of train set)")
    print(f"  Model: {model.__class__.__name__}, LR: {lr}, Epochs: {epochs}, Batch: {batch_size}")
    print(f"  Saving to: {out_dir}")


    for ep in range(1, epochs + 1):
        print(f"\n{run_name} | Epoch {ep}/{epochs}")

        tr_loss = train_epoch(model, train_loader, optimizer, criterion, current_device, scaler, accumulation_steps)
        val_loss, val_psnr, val_ssim = validate(model, val_loader, criterion, current_device)
        scheduler.step(val_loss)

        history.append([ep, tr_loss, val_loss, val_psnr, val_ssim])
        print(f"  Train Loss: {tr_loss:.4f}, Val Loss: {val_loss:.4f}, Val PSNR: {val_psnr:.2f}, Val SSIM: {val_ssim:.4f}")


        if ep % viz_every == 0 or ep == epochs:
            if len(val_loader.dataset) > 0: # Only visualize if val_loader is not empty
                 visualize_results(model, val_loader, current_device, epoch=ep, output_dir_str=str(out_dir))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({'epoch': ep, 'state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()},
                       out_dir / 'best_model.pth')
            print(f"  Saved new best model at epoch {ep} with val_loss: {best_val_loss:.4f}")

        if ep % 10 == 0: # More frequent checkpointing
            torch.save({'epoch': ep, 'state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()},
                       out_dir / f'ckpt_epoch_{ep}.pth')

    hist_np = np.asarray(history)
    if hist_np.size > 0: # Check if history is not empty
        np.savetxt(out_dir / "history.csv", hist_np, delimiter=',',
                   header="epoch,train_loss,val_loss,psnr,ssim", comments='')

        plt.figure(figsize=(18, 5))
        plt.subplot(1, 3, 1); plt.plot(hist_np[:,0], hist_np[:,1], label='Train Loss'); plt.plot(hist_np[:,0], hist_np[:,2], label='Val Loss')
        plt.title(f"Loss ({run_name})"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
        
        plt.subplot(1, 3, 2); plt.plot(hist_np[:,0], hist_np[:,3]); plt.title(f"PSNR ({run_name})")
        plt.xlabel("Epoch"); plt.ylabel("PSNR (dB)"); plt.grid(True)
        
        plt.subplot(1, 3, 3); plt.plot(hist_np[:,0], hist_np[:,4]); plt.title(f"SSIM ({run_name})")
        plt.xlabel("Epoch"); plt.ylabel("SSIM"); plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(out_dir / "training_curves.png")
        plt.close()
    else:
        print(f"Warning: History for {run_name} is empty. No curves plotted.")


    print(f"▶ Finished Run: {run_name}. Best val_loss: {best_val_loss:.4f}")
    return run_name, best_val_loss, (history[-1][3] if history else 0), (history[-1][4] if history else 0) # Return last PSNR/SSIM


# Inference and Model Evaluation

In [73]:
def evaluate_model(model_path, dataloader, device, output_dir="./evaluation"):
    """Evaluate a trained model on a dataset"""
    # Load the model
    model = UNet(in_chans=1, out_chans=1, chans=32, num_pool_layers=4).to(device)
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize metrics
    psnrs = []
    ssims = []
    
    # Process batches
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(tqdm(dataloader, desc="Evaluating")):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Add channel dimension if needed
            if len(inputs.shape) == 3:
                inputs = inputs.unsqueeze(1)
            if len(targets.shape) == 3:
                targets = targets.unsqueeze(1)
            
            # Make predictions
            outputs = model(inputs)
            
            # Calculate metrics
            for j in range(outputs.size(0)):
                output_np = outputs[j, 0].cpu().numpy()
                target_np = targets[j, 0].cpu().numpy()
                
                psnr = calculate_psnr(outputs[j:j+1], targets[j:j+1]).item()
                ssim_val = calculate_ssim(output_np, target_np)
                
                psnrs.append(psnr)
                ssims.append(ssim_val)
            
            # Visualize first batch
            if i == 0:
                fig, axes = plt.subplots(4, 3, figsize=(15, 20))
                fig.suptitle("FastMRI Reconstruction Results", fontsize=16)
                
                for j in range(min(4, outputs.size(0))):
                    # Get images
                    input_img = inputs[j, 0].cpu().numpy()
                    output_img = outputs[j, 0].cpu().numpy()
                    target_img = targets[j, 0].cpu().numpy()
                    
                    # Display input
                    axes[j, 0].imshow(input_img, cmap='gray')
                    axes[j, 0].set_title(f"Input (Undersampled)")
                    axes[j, 0].axis('off')
                    
                    # Display output
                    axes[j, 1].imshow(output_img, cmap='gray')
                    axes[j, 1].set_title(f"Prediction")
                    axes[j, 1].axis('off')
                    
                    # Display target
                    axes[j, 2].imshow(target_img, cmap='gray')
                    axes[j, 2].set_title(f"Ground Truth")
                    axes[j, 2].axis('off')
                    
                    # Add metrics as text
                    axes[j, 1].text(
                        10, 20, 
                        f'PSNR: {psnrs[j]:.2f} dB\nSSIM: {ssims[j]:.4f}',
                        color='white', fontsize=12, 
                        bbox=dict(facecolor='black', alpha=0.5)
                    )
                
                plt.tight_layout()
                plt.subplots_adjust(top=0.95)
                plt.savefig(f"{output_dir}/evaluation_samples.png")
                plt.close()
    
    # Calculate average metrics
    avg_psnr = np.mean(psnrs)
    avg_ssim = np.mean(ssims)
    
    # Print results
    print(f"Average PSNR: {avg_psnr:.2f} dB")
    print(f"Average SSIM: {avg_ssim:.4f}")
    
    # Save metrics
    results = {
        'psnr': psnrs,
        'ssim': ssims,
        'avg_psnr': avg_psnr,
        'avg_ssim': avg_ssim
    }
    
    # Save in text file
    with open(f"{output_dir}/evaluation_results.txt", 'w') as f:
        f.write(f"U-Net Baseline Evaluation Results\n")
        f.write(f"Average PSNR: {avg_psnr:.2f} dB\n")
        f.write(f"Average SSIM: {avg_ssim:.4f}\n")
    
    # Plot histograms of metrics
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(psnrs, bins=20)
    plt.xlabel('PSNR (dB)')
    plt.ylabel('Count')
    plt.title(f'PSNR Histogram (Avg: {avg_psnr:.2f} dB)')
    
    plt.subplot(1, 2, 2)
    plt.hist(ssims, bins=20)
    plt.xlabel('SSIM')
    plt.ylabel('Count')
    plt.title(f'SSIM Histogram (Avg: {avg_ssim:.4f})')
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/metric_histograms.png")
    plt.close()
    
    return results

# Example usage (uncomment to run)
# val_dataset = ProcessedFastMRIDataset(data_dir="./processed_fastmri_data", mode='val', use_processed=True)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)
# results = evaluate_model("./unet_output/best_model.pth", val_loader, device)


# Run

In [74]:
# ─── data loaders (put in a cell ABOVE the pilot sweep) ────────────────────



In [75]:
if __name__ == "__main__":
    # --- Load Datasets ---
    data_root_path = Path("/workspace/fastmri-reconstruction/processed_fastmri_data") # Make sure this path is correct
    print(f"Attempting to load data from: {data_root_path}")
    
    try:
        train_dataset = ProcessedFastMRIDataset(data_dir=data_root_path, mode='train', use_processed=True)
        val_dataset   = ProcessedFastMRIDataset(data_dir=data_root_path, mode='val', use_processed=True)
        print(f"Successfully loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")
    except Exception as e:
        print(f"Error loading datasets: {e}")
        print("Please ensure the data_root_path is correct and data is preprocessed.")
        # Exit or raise if datasets are crucial and not loaded
        raise SystemExit("Dataset loading failed.")


    # --- Define Hyperparameter Configurations for SwinUNet ---
    swin_configs = [
    # Config 1: Solidifying base=40 performance with good VRAM utilization
    {
        'name_suffix': 'base40_ws7_lr1e-4_hds5_bs12_acc1_ep40',
        'base': 40, 'ws': 7, 'lr': 1e-4, 'epochs': 35,
        'batch_size': 12, # Larger direct batch size
        'heads_per_stage': (5,5,5,5), 'frac': 0.5,
        'acc_steps': 1    # Minimal accumulation
    },

    # Config 2: Testing base=48 with optimized batching for more VRAM
    {
        'name_suffix': 'base48_ws7_lr8e-5_hds6_bs8_acc1_ep40',
        'base': 48, 'ws': 7, 'lr': 8e-5, 'epochs': 35,
        'batch_size': 8,  # Good direct batch size for this model size
        'heads_per_stage': (6,6,6,6), 'frac': 0.5,
        'acc_steps': 1
    },

    # Config 3: Pushing to base=64 with a good direct batch size
    {
        'name_suffix': 'base64_ws8_lr8e-5_hds8_bs6_acc1_ep50', # Slightly higher LR than last time's 6e-5 for base64
        'base': 64, 'ws': 8, 'lr': 8e-5, 'epochs': 45, # More epochs for larger model
        'batch_size': 6,  # Increased direct batch size
        'heads_per_stage': (8,8,8,8), 'frac': 0.5,
        'acc_steps': 1    # Prefer acc_steps=1 if VRAM allows bs=6
    },

    # Config 4: Exploring a significantly larger base=80
    {
        'name_suffix': 'base80_ws8_lr6e-5_hds10_bs4_acc2_ep50',
        'base': 80, 'ws': 8, 'lr': 6e-5, 'epochs': 45,
        'batch_size': 4,  # Small direct batch, VRAM permitting
        'heads_per_stage': (10,10,10,10), 'frac': 0.5,
        'acc_steps': 2    # Effective batch size of 8
    }
]

    all_results = []

    for config in swin_configs:
        run_name = f"SwinUNet_Tune_{config['name_suffix']}"
        
        model_lambda = lambda: SwinUNet(
            in_chans=1, 
            out_chans=1, 
            base=config['base'], 
            ws=config['ws'],
            heads_per_stage=config['heads_per_stage']
        )
        
        # It's good practice to ensure datasets are not empty before proceeding
        if len(train_dataset) == 0 or len(val_dataset) == 0:
            print(f"Skipping run {run_name} due to empty dataset(s).")
            continue

        try:
            _, final_loss, final_psnr, final_ssim = run_tuning_experiment(
                run_name=run_name,
                model_fn=model_lambda,
                train_ds=train_dataset,
                val_ds=val_dataset,
                batch_size=config['batch_size'],
                epochs=config['epochs'],
                lr=config['lr'],
                frac=config['frac'], # Use a fraction of data for faster tuning iterations
                viz_every=5, # Visualize every 5 epochs
                device_str=device.type,
                accumulation_steps=config['acc_steps']
            )
            all_results.append({
                'run_name': run_name, 
                'base': config['base'], 
                'ws': config['ws'],
                'heads': config['heads_per_stage'][0], # Assuming same heads for all stages for simplicity in reporting
                'lr': config['lr'],
                'epochs': config['epochs'],
                'batch_size': config['batch_size'],
                'best_val_loss': final_loss, # Note: run_tuning_experiment returns best_val_loss
                'final_psnr': final_psnr,    # This is last epoch's PSNR
                'final_ssim': final_ssim     # This is last epoch's SSIM
            })
        except Exception as e:
            print(f"Error during experiment {run_name}: {e}")
            import traceback
            traceback.print_exc()
            all_results.append({
                'run_name': run_name, 'base': config['base'], 'ws': config['ws'], 
                'heads': config['heads_per_stage'][0], 'lr': config['lr'], 'epochs': config['epochs'],
                'batch_size': config['batch_size'],
                'best_val_loss': float('inf'), 'final_psnr': 0, 'final_ssim': 0, 'error': str(e)
            })

    # --- Display Results ---
    if all_results:
        results_df = pd.DataFrame(all_results)
        results_df = results_df.sort_values(by="best_val_loss", ascending=True)
        print("\n--- Hyperparameter Tuning Results (Sorted by Best Validation Loss) ---")
        
        # Select columns to display, ensure they exist
        cols_to_display = ['run_name', 'base', 'ws', 'heads', 'lr', 'epochs', 'batch_size', 'best_val_loss', 'final_psnr', 'final_ssim']
        if 'error' in results_df.columns: # Add error column if it exists
            cols_to_display.append('error')
        
        # Filter out columns that might not exist if all runs failed early
        existing_cols_to_display = [col for col in cols_to_display if col in results_df.columns]

        if existing_cols_to_display:
             display(results_df[existing_cols_to_display])
        else:
            print("No data to display in results table or columns missing.")

        results_df.to_csv(Path(swin_configs[0].get("out_root_str", "./runs_swin_tuning")) / "swin_unet_tuning_summary.csv", index=False)
        print(f"\nTuning summary saved to {Path(swin_configs[0].get('out_root_str', './runs_swin_tuning')) / 'swin_unet_tuning_summary.csv'}")
    else:
        print("No tuning experiments were completed successfully.")

Attempting to load data from: /workspace/fastmri-reconstruction/processed_fastmri_data
Successfully indexed a total of 973 examples from 61 .pt files in /workspace/fastmri-reconstruction/processed_fastmri_data/train.
Successfully indexed a total of 199 examples from 13 .pt files in /workspace/fastmri-reconstruction/processed_fastmri_data/val.
Successfully loaded 973 training samples and 199 validation samples.

▶ Starting Run: SwinUNet_Tune_base40_ws7_lr1e-4_hds5_bs12_acc1_ep40 – Training on 486 slices (50% of train set)
  Model: SwinUNet, LR: 0.0001, Epochs: 35, Batch: 12
  Saving to: runs_swin_tuning/SwinUNet_Tune_base40_ws7_lr1e-4_hds5_bs12_acc1_ep40

SwinUNet_Tune_base40_ws7_lr1e-4_hds5_bs12_acc1_ep40 | Epoch 1/35


Training:   0%|          | 0/40 [00:00<?, ?it/s]

Validation:   0%|          | 0/17 [00:00<?, ?it/s]

  Train Loss: 0.1920, Val Loss: 0.1021, Val PSNR: 24.20, Val SSIM: 0.5467
  Saved new best model at epoch 1 with val_loss: 0.1021

SwinUNet_Tune_base40_ws7_lr1e-4_hds5_bs12_acc1_ep40 | Epoch 2/35


Training:   0%|          | 0/40 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77c827d818a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1646, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x77c827d818a0>^^
^Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
^    ^self._shutdown_workers()^^
^  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1646, in _shutdown_workers
^^    ^if w.is_alive():^
^ ^^^ ^^^ ^ ^

  Train Loss: 0.0230, Val Loss: 0.0390, Val PSNR: 32.50, Val SSIM: 0.7086
▶ Finished Run: SwinUNet_Tune_base80_ws8_lr6e-5_hds10_bs4_acc2_ep50. Best val_loss: 0.0378

--- Hyperparameter Tuning Results (Sorted by Best Validation Loss) ---


,run_name,base,ws,heads,lr,epochs,batch_size,best_val_loss,final_psnr,final_ssim
2,SwinUNet_Tune_base64_ws8_lr8e-5_hds8_bs6_acc1_...,64,8,8,0.00008,45,6,0.037744,32.720514,0.702686
3,SwinUNet_Tune_base80_ws8_lr6e-5_hds10_bs4_acc2...,80,8,10,0.00006,45,4,0.037797,32.500804,0.708590
0,SwinUNet_Tune_base40_ws7_lr1e-4_hds5_bs12_acc1...,40,7,5,0.00010,35,12,0.039069,32.718679,0.701818
1,SwinUNet_Tune_base48_ws7_lr8e-5_hds6_bs8_acc1_...,48,7,6,0.00008,35,8,0.039387,32.621870,0.687810



Tuning summary saved to runs_swin_tuning/swin_unet_tuning_summary.csv
